# NB2 — Fault Injection

Injects five synthetic fault types (drift, bias, precision_degradation, stuck_at, intermittent_dropout) into normal rows of both train and test splits. Produces a 2×3 visualization panel and a fault distribution table. Saves `data/wadi_faulted.parquet`.

In [1]:
from __future__ import annotations
from datetime import datetime
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)

DATA_DIR    = Path("data")
FIGURES_DIR = DATA_DIR / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED        = 42
TARGET_FAULT_PCT   = 0.20    # fraction of normal rows to receive injected faults
FAULT_DURATION_MIN = 120     # seconds
FAULT_DURATION_MAX = 600     # seconds
MIN_COVERAGE_RATIO = 0.10
BINARY_MAX_UNIQUE  = 5

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

df = pd.read_parquet(DATA_DIR / "wadi_prepared.parquet")
sensor_cols_ref = json.loads((DATA_DIR / "sensor_cols.json").read_text())
SENSOR_COLS = sensor_cols_ref["sensor_cols"]

print(f"Loaded: {df.shape}")
print(f"Sensor columns: {len(SENSOR_COLS)}")


Loaded: (957374, 101)
Sensor columns: 98


## 1. Identify Eligible Sensors

In [2]:
df_normal = df[df["label"] == 0].copy()

eligible   = []
ineligible = []

for col in SENSOR_COLS:
    series = df_normal[col].dropna()
    if series.nunique() <= BINARY_MAX_UNIQUE:
        ineligible.append(col)
        continue
    if len(series) / len(df_normal) < MIN_COVERAGE_RATIO:
        ineligible.append(col)
        continue
    eligible.append(col)

print(f"Eligible sensors:   {len(eligible)}")
print(f"Ineligible sensors: {len(ineligible)} (binary/status or sparse)")


Eligible sensors:   67
Ineligible sensors: 31 (binary/status or sparse)


## 2. Fault Applier Functions

In [3]:
def apply_drift(series: pd.Series, rate_pct_per_min: float) -> pd.Series:
    s = series.copy()
    std = float(s.std())
    rate_per_sec = (rate_pct_per_min / 100.0) * std / 60.0
    drift = np.linspace(0, rate_per_sec * len(s), len(s))
    return (s + drift).astype("float32")

def apply_bias(series: pd.Series, offset_pct: float) -> pd.Series:
    s = series.copy()
    std = float(s.std())
    offset = (offset_pct / 100.0) * std * np.random.choice([-1, 1])
    return (s + offset).astype("float32")

def apply_precision_degradation(series: pd.Series, noise_multiplier: float) -> pd.Series:
    s = series.copy()
    std = float(s.std())
    noise = np.random.normal(0, std * noise_multiplier, size=len(s))
    return (s + noise).astype("float32")

def apply_stuck_at(series: pd.Series, stuck_pct: float) -> pd.Series:
    s = series.copy()
    if np.random.random() < 0.5:
        stuck_val = float(np.nanpercentile(s, stuck_pct * 100))
    else:
        stuck_val = float(np.nanpercentile(s, 100 - stuck_pct * 100))
    return pd.Series(stuck_val, index=s.index, dtype="float32")

def apply_intermittent_dropout(series: pd.Series, dropout_rate: float) -> pd.Series:
    s = series.copy().astype("float32")
    mask = np.random.random(len(s)) < dropout_rate
    s[mask] = np.nan
    return s

FAULT_TYPES = {
    "drift": {
        "description": "Gradual linear deviation from true value",
        "rate_pct_per_min": (0.5, 3.0),
    },
    "bias": {
        "description": "Sudden fixed offset added to all readings",
        "offset_pct": (10.0, 40.0),
    },
    "precision_degradation": {
        "description": "Increased noise around true value",
        "noise_multiplier": (3.0, 8.0),
    },
    "stuck_at": {
        "description": "Sensor freezes at an extreme value outside normal operating range",
        "stuck_pct": (0.01, 0.05),
    },
    "intermittent_dropout": {
        "description": "Random NaN dropouts simulating signal loss",
        "dropout_rate": (0.2, 0.6),
    },
}

FAULT_APPLIERS = {
    "drift":                apply_drift,
    "bias":                 apply_bias,
    "precision_degradation":apply_precision_degradation,
    "stuck_at":             apply_stuck_at,
    "intermittent_dropout": apply_intermittent_dropout,
}
print("Fault appliers defined:", list(FAULT_APPLIERS.keys()))


Fault appliers defined: ['drift', 'bias', 'precision_degradation', 'stuck_at', 'intermittent_dropout']


## 3. Inject Faults

In [4]:
def inject_faults_for_split(
    df_split_normal: pd.DataFrame,
    eligible_sensors: list[str],
    fault_types: dict,
    fault_appliers: dict,
    target_fault_pct: float,
    duration_min: int,
    duration_max: int,
    rng: np.random.Generator,
) -> pd.DataFrame:
    df = df_split_normal.sort_values("timestamp").reset_index(drop=True)

    target_fault_rows     = int(len(df) * target_fault_pct)
    avg_duration          = (duration_min + duration_max) / 2
    faults_per_sensor     = int(target_fault_rows / (len(eligible_sensors) * avg_duration))
    max_faults_per_sensor = int(target_fault_rows / len(eligible_sensors) / avg_duration)
    faults_per_sensor     = max(0, min(faults_per_sensor, max_faults_per_sensor))

    if faults_per_sensor == 0:
        n_sensors_to_use = max(1, int(target_fault_rows / avg_duration))
        eligible_sensors = list(rng.choice(eligible_sensors,
                                           size=min(n_sensors_to_use, len(eligible_sensors)),
                                           replace=False))
        faults_per_sensor = 1

    fault_rows = []

    for sensor in eligible_sensors:
        sensor_data = df[df[sensor].notna()].copy()
        if len(sensor_data) < duration_max * 2:
            continue

        n        = len(sensor_data)
        occupied = np.zeros(n, dtype=bool)

        for _ in range(faults_per_sensor):
            fault_type = rng.choice(list(fault_types.keys()))
            duration   = int(rng.integers(duration_min, duration_max + 1))

            placed = False
            for _ in range(50):
                start_idx = int(rng.integers(0, max(1, n - duration)))
                end_idx   = start_idx + duration
                if not occupied[start_idx:end_idx].any():
                    occupied[start_idx:end_idx] = True
                    placed = True
                    break

            if not placed:
                continue

            window = sensor_data.iloc[start_idx:end_idx].copy()

            cfg = fault_types[fault_type]
            if fault_type == "drift":
                param = float(rng.uniform(*cfg["rate_pct_per_min"]))
                window[sensor] = apply_drift(window[sensor], param)
            elif fault_type == "bias":
                param = float(rng.uniform(*cfg["offset_pct"]))
                window[sensor] = apply_bias(window[sensor], param)
            elif fault_type == "precision_degradation":
                param = float(rng.uniform(*cfg["noise_multiplier"]))
                window[sensor] = apply_precision_degradation(window[sensor], param)
            elif fault_type == "stuck_at":
                param = float(rng.uniform(*cfg["stuck_pct"]))
                window[sensor] = apply_stuck_at(window[sensor], param)
            elif fault_type == "intermittent_dropout":
                param = float(rng.uniform(*cfg["dropout_rate"]))
                window[sensor] = apply_intermittent_dropout(window[sensor], param)

            window["label"]          = 2
            window["fault_type"]     = fault_type
            window["fault_sensor"]   = sensor
            window["fault_start"]    = window["timestamp"].iloc[0]
            window["fault_end"]      = window["timestamp"].iloc[-1]
            window["fault_severity"] = round(float(param), 4)
            fault_rows.append(window)

    if not fault_rows:
        return pd.DataFrame()
    return pd.concat(fault_rows, ignore_index=True)


split_seeds = {"train": RANDOM_SEED, "test": RANDOM_SEED + 1}
fault_dfs = {}
injection_summary = {}

for split in ["train", "test"]:
    rng = np.random.default_rng(split_seeds[split])
    df_split_normal = df[(df["label"] == 0) & (df["split"] == split)].copy()
    print(f"Injecting into {split} ({len(df_split_normal):,} normal rows)...")
    fault_df = inject_faults_for_split(
        df_split_normal  = df_split_normal,
        eligible_sensors = eligible,
        fault_types      = FAULT_TYPES,
        fault_appliers   = FAULT_APPLIERS,
        target_fault_pct = TARGET_FAULT_PCT,
        duration_min     = FAULT_DURATION_MIN,
        duration_max     = FAULT_DURATION_MAX,
        rng              = rng,
    )
    fault_dfs[split] = fault_df
    n_events = 0 if fault_df.empty else fault_df.groupby(["fault_sensor","fault_start"]).ngroups
    injection_summary[split] = {
        "normal_rows": len(df_split_normal),
        "fault_rows":  len(fault_df),
        "fault_events": n_events,
        "fault_pct":   round(len(fault_df) / len(df_split_normal) * 100, 2),
    }
    print(f"  Fault events: {n_events}")
    print(f"  Fault rows:   {len(fault_df):,}  ({injection_summary[split]['fault_pct']:.1f}%)")


Injecting into train (757,890 normal rows)...
  Fault events: 402
  Fault rows:   144,144  (19.0%)
Injecting into test (189,507 normal rows)...
  Fault events: 67
  Fault rows:   23,781  (12.6%)


## 4. Combine & Save

In [5]:
for col in ["fault_type","fault_sensor","fault_start","fault_end","fault_severity"]:
    if col not in df.columns:
        df[col] = np.nan

all_fault_rows = pd.concat([fd for fd in fault_dfs.values() if not fd.empty],
                           ignore_index=True)

# Step 1: Remove normal rows that share a timestamp with any fault row (per split).
# Without this, rolling windows see interleaved normal+fault rows at the same
# timestamp → rolling features average out to near-normal → fault recall collapses.
fault_timestamps_by_split = {}
for split in ["train", "test"]:
    fault_ts = set(
        all_fault_rows[all_fault_rows["split"] == split]["timestamp"].astype(str)
    )
    fault_timestamps_by_split[split] = fault_ts

drop_mask = pd.Series(False, index=df.index)
for split in ["train", "test"]:
    fault_ts = fault_timestamps_by_split[split]
    mask = (
        (df["split"] == split)
        & (df["label"] == 0)
        & (df["timestamp"].astype(str).isin(fault_ts))
    )
    drop_mask |= mask

df_clean = df[~drop_mask].reset_index(drop=True)
print(f"Normal rows removed at fault timestamps: {drop_mask.sum():,}")

# Step 2: Deduplicate fault rows to 1 per (timestamp, split).
# inject_faults_for_split may produce overlapping windows for the same timestamp
# across sensors.  Keep only the first occurrence so each timestamp maps to
# exactly one label=2 row.
fault_deduped = all_fault_rows.drop_duplicates(subset=["timestamp", "split"], keep="first")
print(f"Fault rows before dedup: {len(all_fault_rows):,}  after: {len(fault_deduped):,}")

df_injected = pd.concat([df_clean, fault_deduped], ignore_index=True)
df_injected = df_injected.sort_values("timestamp").reset_index(drop=True)

print(f"\nInjected dataset shape: {df_injected.shape}")
print(f"\nLabel counts:")
for lv, ln in [(0,"normal"),(1,"attack"),(2,"fault")]:
    print(f"  {ln} ({lv}): {(df_injected['label']==lv).sum():>9,}")


Normal rows removed at fault timestamps: 160,995
Fault rows before dedup: 167,925  after: 10,473


/tmp/ipykernel_26080/3850650598.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_injected = pd.concat([df_clean, fault_deduped], ignore_index=True)



Injected dataset shape: (806852, 106)

Label counts:
  normal (0):   786,402
  attack (1):     9,977
  fault (2):    10,473


## 5. Fault Distribution Table

In [6]:
rows = []
for split in ["train","test"]:
    mask = (df_injected["split"]==split) & (df_injected["label"]==2)
    for ft in sorted(FAULT_TYPES.keys()):
        n = (df_injected.loc[mask, "fault_type"] == ft).sum()
        rows.append({"fault_type": ft, "split": split, "rows": n})

dist_df = pd.DataFrame(rows).pivot(index="fault_type", columns="split", values="rows")
dist_df.columns.name = None
dist_df.index.name = "Fault Type"
dist_df["Total"] = dist_df.sum(axis=1)
print(dist_df.to_string())

# Save table as CSV for paper
dist_df.to_csv(DATA_DIR / "results" / "fault_distribution.csv")
print("\nSaved: data/results/fault_distribution.csv")


                       test  train  Total
Fault Type                               
bias                    376   1479   1855
drift                   413   1519   1932
intermittent_dropout    459   2136   2595
precision_degradation   547   1792   2339
stuck_at                139   1613   1752

Saved: data/results/fault_distribution.csv


## 6. Visualization — Fault Type Examples

In [7]:
np.random.seed(RANDOM_SEED)
n = 600
t = np.arange(n)
normal_sig = pd.Series(
    np.sin(2 * np.pi * t / 300) * 2 + 10 + np.random.normal(0, 0.15, n),
    dtype="float32"
)

fault_signals = {
    "Drift":                 apply_drift(normal_sig, rate_pct_per_min=2.0),
    "Bias":                  apply_bias(normal_sig, offset_pct=25.0),
    "Precision Degradation": apply_precision_degradation(normal_sig, noise_multiplier=5.0),
    "Stuck-At":              apply_stuck_at(normal_sig, stuck_pct=0.03),
    "Intermittent Dropout":  apply_intermittent_dropout(normal_sig, dropout_rate=0.4),
}

fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey=False)
axes = axes.flatten()

# Top-left: normal baseline only
axes[0].plot(t, normal_sig.values, lw=1.0, color="steelblue", label="Normal")
axes[0].set_title("Normal (Baseline)", fontsize=11, fontweight="bold")
axes[0].legend(fontsize=8, loc="upper right")

# Remaining 5 panels: normal (gray) under fault (red)
for ax, (title, fault_sig) in zip(axes[1:], fault_signals.items()):
    ax.plot(t, normal_sig.values, lw=1.0, color="steelblue", alpha=0.4,
            linestyle="--", label="Normal", zorder=1)
    ax.plot(t, fault_sig.values, lw=1.0, color="firebrick", label=title,
            zorder=2)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.legend(fontsize=8, loc="upper right")

for ax in axes:
    ax.set_xlabel("Time (s)", fontsize=9)
    ax.set_ylabel("Sensor Value", fontsize=9)
    ax.tick_params(labelsize=8)

fig.suptitle("Injected Fault Types vs. Normal Signal", fontsize=13, fontweight="bold")
fig.tight_layout()

out = FIGURES_DIR / "fault_types_panel.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out}")


Saved: data/results/figures/fault_types_panel.png


In [8]:
out_path = DATA_DIR / "wadi_faulted.parquet"
df_injected.to_parquet(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Shape: {df_injected.shape}")
print(f"Completed: {datetime.now()}")


Saved: data/wadi_faulted.parquet
Shape: (806852, 106)
Completed: 2026-04-19 16:24:15.239891
